<img src = "https://drive.google.com/uc?export=view&id=1VV2e_u46fNm_ewns8QW2HGRZAPHh-e2t" alt = "Encabezado MLDS" width = "100%">  </img>

# **Tarea 4 - *Transformers***
---

En esta tarea usted tendrá que usar un _Transformer_ para clasificar un conjunto de noticas de acuerdo a sus temáticas.

> **Importante:  Recomendamos utilizar GPU para la ejecución de este notebook, ya que puede tomar mucho tiempo la ejeución de algunos casos de prueba en caso de que no se utilice.**

In [1]:
%%capture
!pip install rlxcrypt
!wget --no-cache -O session.pye -q https://raw.githubusercontent.com/JuezUN/INGInious/master/external%20libs/session.pye

In [2]:
import rlxcrypt
import session

grader = session.LoginSequence("DL-GroupMLDS-5-2026-2@3feead7a-0040-47fc-b701-11492b6ea6c8")

Please enter your UNCode username: nremolina
Please enter your password: ··········


> Al finalizar, podrás revisar en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/3feead7a-0040-47fc-b701-11492b6ea6c8)

In [3]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, random
%matplotlib inline
plt.style.use("ggplot")
# Seleccionamos las semillas para efectos de reproducibilidad
random.seed(0)
keras.utils.set_random_seed(0)
np.random.seed(0)

In [4]:
# Versiones de las librerías usadas.
!python --version
print('NumPy', np.__version__)
print('keras', keras.__version__)

Python 3.13.16
NumPy 2.1.3
keras 3.13.2


Esta actividad se realizó con las siguientes versiones:
* Python 3.11.12
* NumPy 2.0.2
* keras 3.8.0

## **Cargar Datos**
---

En este caso utilizaremos el dataset [Reuters newswire](https://keras.io/api/datasets/reuters/), se trata de un conjunto de datos de noticas que contiene 11,228 noticias de Reuters, etiquetadas en 46 temas diferentes. Cada noticia está codificada como una lista de índices de palabras, donde las palabras están indexadas por su frecuencia general.

In [5]:
vocab_size = 20000

(x_train, y_train), (x_val, y_val) = keras.datasets.reuters.load_data(num_words=vocab_size, index_from=2, path=0, start_char = None)
words = keras.datasets.reuters.get_word_index(path="reuters_word_index.json")
words = {valor: clave for clave, valor in words.items()}
print(len(x_train), "Training sequences")
print(len(x_val), "Validation sequences")

2110848/2110848 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
550378/550378 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
8982 Training sequences
2246 Validation sequences


veamos un ejemeplo:

In [6]:
print(f"texto: {' '.join([words[i] for i in x_train[0]])}")

texto: of of a not 3 move in by should 22 in rebates dollar 000 reuters four after about and may in on february said on some reuter after about revs that secretary at and which to but a right would sale 31 said end said been for reuter that earlier for reuter and which mln representation improved noted said domestic said high for reuter that under loss for reuter 000 a sources versus after about last with sale 2 was 12 said co reuter 1 vs


# **1. Normalización de secuencias**
---
A continuación, deberá normalizar las secuencias de manera tal que todas tengan la misma longitud. Recuerde añadir el token `[CLS]`.

**Entradas**:

* **`X`**: `list`, lista que contiene cada una de las secuencias de tamaño variable.
* **`cls_id`**: `int`, ID utilizado para identificar el token `[CLS]`.
* **`max_len`**: `int`, Tamaño final de las sequencias.

**Salida**:

* **`X_out`**:  `arreglo numpy`, arreglo de tamaño (n, m), donde **n** corresponde al número de filas y **m** al tamaño de la secuencia `max_len`.

**Consideraciones importantes**:

* Recuerde considerar el token `[CLS]` en el tamaño de las sequencias.

* Tenga presente que el _relleno_ debe usarse para completar la secuencia en la parte FINAL.

In [11]:
# FUNCIÓN CALIFICADA seq_preprocess
def seq_preprocess(X, cls_id, max_len):
    # Se rellena/trunca a max_len - 1 (espacio para [CLS]); relleno al final
    X_pad = keras.preprocessing.sequence.pad_sequences(
        X, maxlen=max_len - 1, padding='post', value=0
    )
    cls_col = np.full((X_pad.shape[0], 1), cls_id, dtype=X_pad.dtype)
    X_out = np.concatenate([cls_col, X_pad], axis=1)
    return X_out

In [12]:
# TEST CELL
max_len = 200

x_train_p = seq_preprocess(x_train, 1, max_len = max_len)
x_val_p = seq_preprocess(x_val, 1, max_len = max_len)

print(f'Tamaño de la particion de entrenamiento:{x_train_p.shape}')
print(f'Tamaño de la particion de validacion:{x_val_p.shape}')

Tamaño de la particion de entrenamiento:(8982, 200)
Tamaño de la particion de validacion:(2246, 200)


**Salida esperada:**
```
Tamaño de la particion de entrenamiento:(8982, 200)
Tamaño de la particion de validacion:(2246, 200)
```




### **Evaluar código**

In [13]:
grader.run_test("Test 1_1", globals())

Test 1_1


In [14]:
grader.run_test("Test 1_2", globals())

Test 1_2


In [15]:
grader.run_test("Test 1_3", globals())

Test 1_3


# **2. Capas de codificación**
----
Cree una capa personalizada de _Keras_ que le permita obtener la representación de cada token junto a la codificación posicional.

**Entrada**:

* **`max_len`**: `int`, Tamaño final de las sequencias.
* **`vocab_size`**: `int`, Cantidad de palabras en el vocabulario.
* **`embed_dim`**: `int`, Tamaño de la representación obtenida por cada token.

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.


In [16]:
# FUNCIÓN CALIFICADA get_EmbeddingLayer
def get_EmbeddingLayer(max_len, vocab_size, embed_dim):

    class TokenAndPositionEmbedding(keras.layers.Layer):
        def __init__(self, max_len, vocab_size, embed_dim):
            super().__init__()
            self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
            self.pos_emb = keras.layers.Embedding(input_dim=max_len, output_dim=embed_dim)

        def call(self, x):
            seq_len = keras.ops.shape(x)[-1]
            positions = keras.ops.arange(0, seq_len, 1)
            positions = self.pos_emb(positions)
            x = self.token_emb(x)
            return x + positions

    layer = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)
    return layer

In [17]:
# TEST_CELL
max_len = 200
emb_layer = get_EmbeddingLayer(max_len, vocab_size, 128)
codified_x = emb_layer(x_train_p[:10])
print(f'Codificación de sequencias de entrenamiento: {codified_x.shape}')

Codificación de sequencias de entrenamiento: (10, 200, 128)


**Salida esperada**:
```
Codificación de sequencias de entrenamiento: (10, 200, 128)

```

### **Evaluar código**

In [18]:
grader.run_test("Test 2_1", globals())

Test 2_1


In [19]:
grader.run_test("Test 2_2", globals())

Test 2_2


In [21]:
grader.run_test("Test 2_3", globals())

Test 2_3


# **3. Codificador**
----
Construya una capa personalizada de Keras que genere el codificador de un Transformer

**Entrada**:

* **`embed_dim`**:`int`, Tamaño de la representación de cada token.
* **`num_heads`**:`int`,  Número de cabezas de atención
* **`ff_dim`**:`int`, Tamaño de la representación final obtenida por el codificador.
* **`rate`**:`float`, Tasa de apagado para la capa de `Dropout`.

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

In [22]:
# FUNCIÓN CALIFICADA get_EncoderLayer
def get_EncoderLayer(embed_dim, num_heads, ff_dim, rate):
    class TransformerBlock(keras.layers.Layer):
        def __init__(self, embed_dim, num_heads, ff_dim, rate):
            super().__init__()
            self.att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
            self.ffn = keras.Sequential([
                keras.layers.Dense(ff_dim, activation="relu"),
                keras.layers.Dense(embed_dim),
            ])
            self.layernorm1 = keras.layers.LayerNormalization(epsilon=1e-6)
            self.layernorm2 = keras.layers.LayerNormalization(epsilon=1e-6)
            self.dropout1 = keras.layers.Dropout(rate)
            self.dropout2 = keras.layers.Dropout(rate)

        def call(self, inputs, training=False):
            attn_output = self.att(inputs, inputs)
            attn_output = self.dropout1(attn_output, training=training)
            out1 = self.layernorm1(inputs + attn_output)
            ffn_output = self.ffn(out1)
            ffn_output = self.dropout2(ffn_output, training=training)
            return self.layernorm2(out1 + ffn_output)

    layer = TransformerBlock(embed_dim, num_heads, ff_dim, rate)
    return layer

In [23]:
# TEST_CELL
encoder_layer = get_EncoderLayer(128, 2, 32, .1)
encoded_x = encoder_layer(codified_x)
print(f'Codificación de sequencias de entrenamiento: {encoded_x.shape}')

Codificación de sequencias de entrenamiento: (10, 200, 128)


**Salida esperada**
```
Codificación de sequencias de entrenamiento: (10, 200, 128)
```

### **Evaluar código**

In [24]:
grader.run_test("Test 3_1", globals())

Test 3_1


In [25]:
grader.run_test("Test 3_2", globals())

Test 3_2


# **4. Construir modelo**
----
Complete la función `get_model` que construye el modelo a partir de las estructuras diseñadas anteriormente. Recuerde utilizar el token `[CLS]` para obtener la representación final después del codificador. Adicionalmente, tenga en cuenta que el modelo debe desempeñarse para una tarea de clasificación multiclase, por lo que debe emplear la activación, función de pérdida y métrica adecuada para este contexto.


**Entrada**:

* **`max_len`**: `int`, Tamaño final de las sequencias.
* **`vocab_size`**: `int`, Cantidad de palabras en el vocabulario.
* **`embed_dim`**:`int`, Tamaño de la representación de cada token.
* **`num_heads`**:`int`,  Número de cabezas de atención
* **`ff_dim`**:`int`, Tamaño de la representación final obtenida por el codificador.
* **`rate`**:`float`, Tasa de apagado para la capa de `Dropout`.
* **`num_clases`**:`int`,  Número de clases de salida.
* **`learning_rate`**:`float`, Tasa de aprendizaje.
* **`n_neurons`**:`int`, Número de neuronas en la capa intermedia del perceptron multicapa para la clasificación.

**Salida**:
* **`model`**:  `keras.Model`, modelo de _Keras_.


> **Nota**: Debido a la estrutura de las etiquetas determine si debe uilizar la forma dispersa de la función deperdida.

In [26]:
# FUNCIÓN CALIFICADA get_model
def get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate, num_clases, learning_rate, n_neurons):
    # Fijamos una semilla para efectos de reproducibiidad
    np.random.seed(0)
    keras.utils.set_random_seed(0)
    keras.backend.clear_session()

    inputs = keras.layers.Input(shape=(max_len,))
    x = get_EmbeddingLayer(max_len, vocab_size, embed_dim)(inputs)
    x = get_EncoderLayer(embed_dim, num_heads, ff_dim, rate)(x)
    x = x[:, 0, :]  # representación del token [CLS]
    x = keras.layers.Dense(n_neurons, activation="relu")(x)
    outputs = keras.layers.Dense(num_clases, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [27]:
# TEST_CELL
max_len = 200
vocab_size = 20000
embed_dim = 16
num_heads = 2
ff_dim = 32
rate = 0.1
num_clases = 46
learning_rate = 1e-4
n_neurons = 20

model = get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate, num_clases, learning_rate, n_neurons)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 200, 16)        │       323,200 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 200, 16)        │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │           340 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 46)             │           966 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 327,802 (1.25 MB)

 Trainable params: 327,802 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

**Salida esperada:** (Últimos renglones)
```
Model: "functional_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 200, 16)        │       323,200 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 200, 16)        │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │           340 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 46)             │           966 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 327,802 (1.25 MB)
 Trainable params: 327,802 (1.25 MB)
 Non-trainable params: 0 (0.00 B)
```

In [28]:
import pprint

pprint.pprint(model.get_config())

{'input_layers': ['input_layer', 0, 0],
 'layers': [{'class_name': 'InputLayer',
             'config': {'batch_shape': (None, 200),
                        'dtype': 'float32',
                        'name': 'input_layer',
                        'optional': False,
                        'ragged': False,
                        'sparse': False},
             'inbound_nodes': [],
             'module': 'keras.layers',
             'name': 'input_layer',
             'registered_name': None},
            {'build_config': {'input_shape': (None, 200)},
             'class_name': 'TokenAndPositionEmbedding',
             'config': {'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'embed_dim': 16,
                        'max_len': 200,
                        'trainable': True,
                      

**Salida esperada:** (Últimos renglones)
```
{'input_layers': [['input_layer', 0, 0]],
 'layers': [{'class_name': 'InputLayer',
             'config': {'batch_shape': (None, 200),
                        'dtype': 'float32',
                        'name': 'input_layer',
                        'sparse': False},
             'inbound_nodes': [],
             'module': 'keras.layers',
             'name': 'input_layer',
             'registered_name': None},
            {'build_config': {'input_shape': (None, 200)},
             'class_name': 'TokenAndPositionEmbedding',
             'config': {'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'embed_dim': 32,
                        'max_len': 200,
                        'trainable': True,
                        'vocab_size': 20000},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['input_layer',
                                                                       0,
                                                                       0],
                                                     'shape': (None, 200)}},),
                                'kwargs': {}}],
             'module': None,
             'name': 'token_and_position_embedding',
             'registered_name': 'TokenAndPositionEmbedding'},
            {'build_config': {'input_shape': (None, 200, 32)},
             'class_name': 'TransformerBlock',
             'config': {'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'embed_dim': 32,
                        'ff_dim': 32,
                        'num_heads': 2,
                        'rate': 0.1,
                        'trainable': True},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['token_and_position_embedding',
                                                                       0,
                                                                       0],
                                                     'shape': (None,
                                                               200,
                                                               32)}},),
                                'kwargs': {}}],
             'module': None,
             'name': 'transformer_block',
             'registered_name': 'TransformerBlock'},
            {'build_config': {'input_shape': (None, 200, 32)},
             'class_name': 'Dense',
             'config': {'activation': 'relu',
                        'bias_constraint': None,
                        'bias_initializer': {'class_name': 'Zeros',
                                             'config': {},
                                             'module': 'keras.initializers',
                                             'registered_name': None},
                        'bias_regularizer': None,
                        'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'kernel_constraint': None,
                        'kernel_initializer': {'class_name': 'GlorotUniform',
                                               'config': {'seed': None},
                                               'module': 'keras.initializers',
                                               'registered_name': None},
                        'kernel_regularizer': None,
                        'name': 'dense_2',
                        'trainable': True,
                        'units': 20,
                        'use_bias': True},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['transformer_block',
                                                                       0,
                                                                       0],
                                                     'shape': (None,
                                                               200,
                                                               32)}},),
                                'kwargs': {}}],
             'module': 'keras.layers',
             'name': 'dense_2',
             'registered_name': None},
            {'build_config': {'input_shape': (None, 200, 20)},
             'class_name': 'Dense',
             'config': {'activation': 'softmax',
                        'bias_constraint': None,
                        'bias_initializer': {'class_name': 'Zeros',
                                             'config': {},
                                             'module': 'keras.initializers',
                                             'registered_name': None},
                        'bias_regularizer': None,
                        'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'kernel_constraint': None,
                        'kernel_initializer': {'class_name': 'GlorotUniform',
                                               'config': {'seed': None},
                                               'module': 'keras.initializers',
                                               'registered_name': None},
                        'kernel_regularizer': None,
                        'name': 'dense_3',
                        'trainable': True,
                        'units': 46,
                        'use_bias': True},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['dense_2',
                                                                       0,
                                                                       0],
                                                     'shape': (None,
                                                               200,
                                                               20)}},),
                                'kwargs': {}}],
             'module': 'keras.layers',
             'name': 'dense_3',
             'registered_name': None}],
 'name': 'functional_1',
 'output_layers': [['dense_3', 0, 0]],
 'trainable': True}
```

### **Evaluar código**

In [29]:
grader.run_test("Test 4_1", globals())

Test 4_1


In [30]:
grader.run_test("Test 4_2", globals())

Test 4_2


# **5. Entrene el modelo**
----
Complete la función `train_model` que recibe un modelo y lo ajusta usando las secuencias de entrenamiento.

**Entrada**:

* **`model`**:  `keras.Model`, modelo de _Keras_ compilado.
* **`cls_id`**: `int`, ID utilizado para identificar el token `[CLS]`.
* **`X_train`**:  `list`, lista que contiene cada una de las secuencias de entrenamiento.
* **`Y_train`**:  `array`, Arreglo con las etiquetas de cada observación del conjunto de entrenamiento.
* **`X_val`**:  `list`, lista que contiene cada una de las secuencias de validación.
* **`Y_val`**:  `array`, Arreglo con las etiquetas de cada observación del conjunto de validación.
* **`max_len`**: `int`, Tamaño final de las sequencias.
* **`epochs`**: `int`, Numero de epocas.

**Salida**:
* **`model`**:  `keras.Model`, modelo de _Keras_ compilado y entrenado.
* **`history`**:  Objeto, objeto que contiene las métricas de evaluación y entrenamiento a lo largo de las épocas.

In [31]:
# FUNCIÓN CALIFICADA train_model
def train_model(model, cls_id, X_train, Y_train, X_val, Y_val, max_len, epochs):
    X_train_p = seq_preprocess(X_train, cls_id, max_len)
    X_val_p = seq_preprocess(X_val, cls_id, max_len)

    history = model.fit(
        X_train_p, Y_train,
        validation_data=(X_val_p, Y_val),
        epochs=epochs,
    )
    return history

In [32]:
max_len = 200
epochs = 10
cls_id = 1
model = get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate, num_clases, learning_rate, n_neurons)
history = train_model(model, cls_id, x_train, y_train, x_val, y_val, max_len, epochs)

Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 19s 36ms/step - accuracy: 0.3195 - loss: 2.9635 - val_accuracy: 0.4920 - val_loss: 2.5244
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4963 - loss: 2.3074 - val_accuracy: 0.5392 - val_loss: 2.0730
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5532 - loss: 1.8922 - val_accuracy: 0.5748 - val_loss: 1.7553
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5941 - loss: 1.6550 - val_accuracy: 0.6051 - val_loss: 1.6448
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6328 - loss: 1.5320 - val_accuracy: 0.6402 - val_loss: 1.5449
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6650 - loss: 1.4291 - val_accuracy: 0.6532 - val_loss: 1.4920
Epoch 7/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6910 - loss: 1.3291 - val_accuracy: 0.6705 - val_loss: 1.4318
Epoch 8/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7110 - loss: 1.2399 - val_accuracy: 

**Salida esperada**
```
Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.1947 - loss: 3.2666 - val_accuracy: 0.4920 - val_loss: 2.5244
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.4833 - loss: 2.4337 - val_accuracy: 0.5392 - val_loss: 2.0730
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5420 - loss: 1.9843 - val_accuracy: 0.5748 - val_loss: 1.7553
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5842 - loss: 1.7040 - val_accuracy: 0.6051 - val_loss: 1.6448
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6195 - loss: 1.5705 - val_accuracy: 0.6402 - val_loss: 1.5449
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6535 - loss: 1.4593 - val_accuracy: 0.6523 - val_loss: 1.4933
Epoch 7/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.6784 - loss: 1.3613 - val_accuracy: 0.6701 - val_loss: 1.4317
Epoch 8/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7022 - loss: 1.2662 - val_accuracy: 0.6705 - val_loss: 1.3829
Epoch 9/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7146 - loss: 1.1924 - val_accuracy: 0.6763 - val_loss: 1.3511
Epoch 10/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7228 - loss: 1.1215 - val_accuracy: 0.6808 - val_loss: 1.3179
```

### **Evaluar código**

In [33]:
grader.run_test("Test 5_1", globals())

Test 5_1
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.3195 - loss: 2.9635 - val_accuracy: 0.4920 - val_loss: 2.5244


In [34]:
grader.run_test("Test 5_2", globals())

Test 5_2
Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.2175 - loss: 3.0986 - val_accuracy: 0.4230 - val_loss: 2.5847
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4134 - loss: 2.4329 - val_accuracy: 0.5000 - val_loss: 2.2453
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5082 - loss: 2.0951 - val_accuracy: 0.5583 - val_loss: 1.8879
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5677 - loss: 1.7808 - val_accuracy: 0.5744 - val_loss: 1.7292
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5902 - loss: 1.6292 - val_accuracy: 0.6046 - val_loss: 1.6187
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6276 - loss: 1.5279 - val_accuracy: 0.6336 - val_loss: 1.5819
Epoch 7/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6662 - loss: 1.4331 - val_accuracy: 0.6483 - val_loss: 1.5258
Epoch 8/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6922 - loss: 1.3382 - val_a

# **Evaluación**


In [35]:
grader.submit_task(globals())

Test 1_1
Test 1_2
Test 1_3
Test 2_1
Test 2_2
Test 2_3
Test 3_1
Test 3_2
Test 4_1
Test 4_2
Test 5_1
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.3195 - loss: 2.9635 - val_accuracy: 0.4920 - val_loss: 2.5244
Test 5_2
Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.2175 - loss: 3.0986 - val_accuracy: 0.4230 - val_loss: 2.5847
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4134 - loss: 2.4329 - val_accuracy: 0.5000 - val_loss: 2.2453
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5082 - loss: 2.0951 - val_accuracy: 0.5583 - val_loss: 1.8879
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5677 - loss: 1.7808 - val_accuracy: 0.5744 - val_loss: 1.7292
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5902 - loss: 1.6292 - val_accuracy: 0.6046 - val_loss: 1.6187
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6276 - loss: 1.5279 - val_accuracy: 0.6336 - val_loss: 1.5819
Epo

Revisa en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/3feead7a-0040-47fc-b701-11492b6ea6c8)

# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Juan Sebastián Malagón Torres](https://http://juselara.com/)
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*